# Fresh vs Rotten Fruit Classification — Dataset Exploration

This notebook explores the dataset used for the Fresh vs Rotten Fruit Classification project.

## Dataset

The dataset contains six classes:

- Fresh Apple
- Rotten Apple
- Fresh Banana
- Rotten Banana
- Fresh Orange
- Rotten Orange

The dataset is divided into training and testing directories.

In [5]:
import kagglehub
from pathlib import Path

# Authenticate with Kaggle
kagglehub.login()

# Download/access the dataset
dataset_path = kagglehub.dataset_download(
    "sriramr/fruits-fresh-and-rotten-for-classification"
)

# Actual dataset directory
DATASET_ROOT = Path(dataset_path) / "dataset"

TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Dataset root:", DATASET_ROOT)
print("Train directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)

Using Colab cache for faster access to the 'fruits-fresh-and-rotten-for-classification' dataset.
Dataset root: /kaggle/input/fruits-fresh-and-rotten-for-classification/dataset
Train directory: /kaggle/input/fruits-fresh-and-rotten-for-classification/dataset/train
Test directory: /kaggle/input/fruits-fresh-and-rotten-for-classification/dataset/test
Kaggle credentials set.
Kaggle credentials successfully validated.


## 1. Dataset Structure and Class Distribution

In [6]:
def count_images(folder):
    return sum(
        1
        for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


print("TRAINING SET")
print("============")

train_total = 0

for class_dir in sorted(TRAIN_DIR.iterdir()):
    if class_dir.is_dir():
        count = count_images(class_dir)
        train_total += count
        print(f"{class_dir.name:20s}: {count}")

print(f"{'TOTAL':20s}: {train_total}")


print("\nTEST SET")
print("========")

test_total = 0

for class_dir in sorted(TEST_DIR.iterdir()):
    if class_dir.is_dir():
        count = count_images(class_dir)
        test_total += count
        print(f"{class_dir.name:20s}: {count}")

print(f"{'TOTAL':20s}: {test_total}")

print("\nOVERALL TOTAL:", train_total + test_total)

TRAINING SET
freshapples         : 1693
freshbanana         : 1581
freshoranges        : 1466
rottenapples        : 2342
rottenbanana        : 2224
rottenoranges       : 1595
TOTAL               : 10901

TEST SET
freshapples         : 395
freshbanana         : 381
freshoranges        : 388
rottenapples        : 601
rottenbanana        : 530
rottenoranges       : 403
TOTAL               : 2698

OVERALL TOTAL: 13599


## 2. Class Distribution

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

class_counts = {}

for class_dir in sorted(TRAIN_DIR.iterdir()):
    if class_dir.is_dir():
        class_counts[class_dir.name] = count_images(class_dir)

print("Training class distribution:")
for class_name, count in class_counts.items():
    print(f"{class_name}: {count}")

## 3. Training Class Distribution Visualization

In [ ]:
import matplotlib.pyplot as plt

class_names = list(class_counts.keys())
counts = list(class_counts.values())

plt.figure(figsize=(10, 6))
plt.bar(class_names, counts)

plt.title("Training Dataset Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Fresh vs Rotten Distribution

In [ ]:
fresh_count = (
    class_counts["freshapples"]
    + class_counts["freshbanana"]
    + class_counts["freshoranges"]
)

rotten_count = (
    class_counts["rottenapples"]
    + class_counts["rottenbanana"]
    + class_counts["rottenoranges"]
)

total_count = fresh_count + rotten_count

fresh_percentage = (fresh_count / total_count) * 100
rotten_percentage = (rotten_count / total_count) * 100

print("Fresh images :", fresh_count)
print("Rotten images:", rotten_count)
print("Total images :", total_count)

print(f"\nFresh percentage : {fresh_percentage:.2f}%")
print(f"Rotten percentage: {rotten_percentage:.2f}%")

In [ ]:
labels = ["Fresh", "Rotten"]
counts = [fresh_count, rotten_count]

plt.figure(figsize=(8, 5))
plt.bar(labels, counts)

plt.title("Fresh vs Rotten Training Distribution")
plt.xlabel("Category")
plt.ylabel("Number of Images")

plt.tight_layout()
plt.show()

## 4. Fresh vs Rotten Distribution — Observation

The training dataset contains 10,901 images in total.

- Fresh fruits: 4,740 images (43.48%)
- Rotten fruits: 6,161 images (56.52%)

The training dataset therefore contains more rotten-fruit images than fresh-fruit images. This indicates some imbalance at the fresh-versus-rotten level.

We will keep this distribution in mind during model training and evaluation. Accuracy alone will not be used to judge the model; precision, recall, F1-score, and the confusion matrix will also be considered.

## 5. Random Sample Images from Each Class

In [ ]:
from PIL import Image
import random
import matplotlib.pyplot as plt

random.seed(42)

class_dirs = sorted(
    [class_dir for class_dir in TRAIN_DIR.iterdir() if class_dir.is_dir()]
)


def prepare_image(image_path):
    with Image.open(image_path) as image:
        if image.mode == "RGBA":
            background = Image.new("RGB", image.size, "white")
            background.paste(image, mask=image.getchannel("A"))
            return background.copy()

        return image.convert("RGB")


fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, class_dir in zip(axes.ravel(), class_dirs):
    image_files = [
        path
        for path in class_dir.iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    image_path = random.choice(image_files)
    image = prepare_image(image_path)

    ax.imshow(image)
    ax.set_title(class_dir.name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Inspecting RGBA Transparency

In [ ]:
from PIL import Image
import numpy as np

rgba_image_path = None

for class_dir in sorted(TRAIN_DIR.iterdir()):
    if not class_dir.is_dir():
        continue

    for image_path in class_dir.iterdir():
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        with Image.open(image_path) as image:
            if image.mode == "RGBA":
                rgba_image_path = image_path
                break

    if rgba_image_path is not None:
        break


print("Selected RGBA image:")
print(rgba_image_path)

with Image.open(rgba_image_path) as image:
    image = image.copy()

alpha = np.array(image.getchannel("A"))

print("\nAlpha channel:")
print("Minimum:", alpha.min())
print("Maximum:", alpha.max())
print("Transparent pixels:", np.sum(alpha == 0))
print("Non-transparent pixels:", np.sum(alpha > 0))
print("Total pixels:", alpha.size)

## 7. RGBA Transparency Check Across the Dataset

In [ ]:
from PIL import Image
import numpy as np

rgba_count = 0
transparent_image_count = 0
transparent_pixel_count = 0

for split_dir in [TRAIN_DIR, TEST_DIR]:
    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir():
            continue

        for image_path in class_dir.iterdir():
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            with Image.open(image_path) as image:
                if image.mode != "RGBA":
                    continue

                rgba_count += 1

                alpha = np.array(image.getchannel("A"))

                if np.any(alpha < 255):
                    transparent_image_count += 1
                    transparent_pixel_count += np.sum(alpha < 255)

print("RGBA images checked:", rgba_count)
print("Images containing transparent pixels:", transparent_image_count)
print("Total transparent pixels:", transparent_pixel_count)